# 🎙️ VoiceNO — Open-Vocabulary Silent Speech Evaluation & Contextual Disambiguation

**Tagline**: *You have the right to remain silent.*

This Google Colab notebook provides an end-to-end testing environment for **VoiceNO**'s open-vocabulary silent speech recognition, N-best hypothesis preservation, contextual rescoring, and uncertainty handling.

### Core Research Question:
> *Can VoiceNO preserve multiple plausible interpretations from visual speech and use personalization and context to recover what the user actually intended without a closed command vocabulary?*

### What this notebook tests:
1. **Condition A:** Generic Open-Vocabulary Recognition (Auto-AVSR Baseline)
2. **Condition B:** Personalized Open-Vocabulary Recognition (LoRA Adapter)
3. **Condition C:** Personalized + N-best Hypotheses (Oracle Top-3 / Top-5 Recovery)
4. **Condition D:** Personalized + N-best + Contextual / Linguistic Rescoring

In [ ]:
# 1. Environment & Hardware Verification
!nvidia-smi || echo "Running on CPU mode"

In [ ]:
# 2. Clone Repository & Install Dependencies
!git clone https://github.com/your-username/VoiceNo.git /content/VoiceNo || true
%cd /content/VoiceNo
!pip install -q torch torchvision torchaudio
!pip install -q opencv-python mediapipe sentencepiece jiwer gradio

In [ ]:
# 3. Verify Auto-AVSR Checkpoint
import os
from pathlib import Path

ckpt_path = Path("checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth")
if not ckpt_path.is_file():
    print("Downloading Auto-AVSR base checkpoint...")
    # Provide direct download link or Google Drive copy here
else:
    print(f"Checkpoint verified: {ckpt_path} ({ckpt_path.stat().st_size / (1024*1024):.1f} MB)")

In [ ]:
# 4. Run Open-Vocabulary Multi-Condition Evaluation (A, B, C, D)
!python experiments/run_open_vocab_eval.py --user default

In [ ]:
# 5. Interactive Demo: Inspect N-Best Hypotheses and Contextual Reranking
from voiceno.models.auto_avsr import AutoAVSRModel
from voiceno.context.rescorer import ContextRescorer, TaskContext
import json

model = AutoAVSRModel()
model.load_checkpoint()

test_video = "tests/benchmark_sample.mp4"
result = model.transcribe(test_video, use_beam_search=True, nbest=5)

print("=== VISUAL MODEL OUTPUT ===")
print(f"Top-1 Candidate: {result['transcription']}")
print(f"Confidence: {result['confidence_level']} (Margin: {result['score_margin']})")
print("\nPreserved N-Best Visual Hypotheses:")
for i, h in enumerate(result['hypotheses'][:5]):
    print(f"  [{i+1}] Score: {h['score']:.2f} | Prob: {h.get('normalized_prob', 0.0)*100:.1f}% | Text: {h['text']}")

# Apply Task Context (e.g. user is browsing about future world development)
context = TaskContext(
    active_app="browser",
    domain_keywords=["future", "country", "world", "better"],
)
rescorer = ContextRescorer()
rescored = rescorer.rescore_hypotheses(result['hypotheses'], context=context)

print("\n=== CONTEXT-RERANKED OUTPUT ===")
for i, h in enumerate(rescored[:5]):
    print(f"  [{i+1}] Combined Score: {h['rescored_score']:.2f} (Visual: {h['visual_score']:.2f}, Lang: {h['linguistic_score']:.2f}, Ctx: {h['context_score']:.2f}) | Text: {h['text']}")

In [ ]:
# 6. Launch VoiceNO! Web UI with N-Best Table & Clarification UI
from app.ui import build_ui

demo = build_ui()
demo.launch(share=True)